# OpenTelemetry Telemetry Explorer

This notebook demonstrates the out-of-core processing and visualization capabilities
of the `cybersec.observability` module for OpenTelemetry data stored in Parquet format.

## Getting Started

**Important**: This notebook is mounted read-only from a ConfigMap. To edit and run:
1. Copy to your home directory: `cp ~/sample-notebooks/OTel_Telemetry_Explorer.ipynb ~/`
2. Open the copy from the file browser

Or run this in a cell:
```python
import shutil
shutil.copy('/home/jovyan/sample-notebooks/OTel_Telemetry_Explorer.ipynb', '/home/jovyan/')
```

## Prerequisites

- Dask cluster running (local or distributed)
- S3 bucket for data storage (`OTEL_DATA_PATH` environment variable)
- Required packages: `dask`, `holoviews`, `panel`, `networkx`, `s3fs`

In [ ]:
# Standard imports
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings('ignore')

# Visualization extensions
import holoviews as hv
import panel as pn
hv.extension('bokeh')
pn.extension()

## 1. Connect to Dask Cluster

Connect to either a local or distributed Dask cluster for out-of-core processing.

In [ ]:
import os
from dask.distributed import Client

# Connect to remote cluster (from environment) or create local
scheduler_address = os.getenv('DASK_SCHEDULER_ADDRESS')
if scheduler_address:
    print(f"Connecting to remote scheduler: {scheduler_address}")
    client = Client(scheduler_address)
else:
    print("Creating local cluster (set DASK_SCHEDULER_ADDRESS for remote)")
    client = Client(n_workers=4, threads_per_worker=2, memory_limit='4GB')

print(f"Dashboard: {client.dashboard_link}")
client

## 2. Initialize OTel Dataset

Create the dataset reader with S3 credentials for MinIO.

In [ ]:
import os
from cybersec.observability import OTelDataset

# S3 configuration from environment
# For AWS: uses IAM role credentials automatically (no keys needed)
# For MinIO: set S3_ENDPOINT, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY
storage_options = {}

# Add credentials only if explicitly set (for non-IAM environments)
if os.getenv('AWS_ACCESS_KEY_ID'):
    storage_options['key'] = os.getenv('AWS_ACCESS_KEY_ID')
    storage_options['secret'] = os.getenv('AWS_SECRET_ACCESS_KEY')

# Add endpoint_url only for MinIO/non-AWS
s3_endpoint = os.getenv('S3_ENDPOINT')
if s3_endpoint:
    storage_options['endpoint_url'] = s3_endpoint

# Data path from environment (set by JupyterHub)
# Default: s3://cybersec-dask-data/otel/
otel_data_path = os.getenv('OTEL_DATA_PATH', 's3://cybersec-dask-data/otel/')

dataset = OTelDataset(
    base_path=otel_data_path,
    storage_options=storage_options if storage_options else None,
    dask_client=client,
)

print(f"Dataset initialized at: {otel_data_path}")
print(f"Using IAM role: {not bool(storage_options)}")

## 3. Generate Synthetic Data (Idempotent)

Generate synthetic OTel data for demonstration. Uses `if_not_exists=True` to skip 
generation if data already exists, preventing duplicate datasets in S3.

In [ ]:
from cybersec.observability import OTelWriter

# Create writer with same storage options
writer = OTelWriter(
    base_path=otel_data_path,
    storage_options=storage_options if storage_options else None,
)

# Check current data status
print("Checking existing data...")
for data_type in ['spans', 'metrics', 'logs']:
    stats = writer.get_data_stats(data_type)
    if stats['file_count'] > 0:
        print(f"  {data_type}: {stats['file_count']} files ({stats['total_size_bytes']/1024/1024:.1f} MB)")
    else:
        print(f"  {data_type}: no data")

# Generate synthetic data (idempotent - skips if data exists)
print("\nGenerating synthetic spans (if needed)...")
span_files = writer.write_synthetic_spans(
    count=50_000,       # 50K spans
    services=5,
    duration_hours=2,
    if_not_exists=True,  # Skip if data already exists
)
if span_files:
    print(f"Written {len(span_files)} span files")

print("\nGenerating synthetic metrics (if needed)...")
metric_files = writer.write_synthetic_metrics(
    count=10_000,       # 10K metric points
    duration_hours=2,
    if_not_exists=True,
)
if metric_files:
    print(f"Written {len(metric_files)} metric files")

print("\nGenerating synthetic logs (if needed)...")
log_files = writer.write_synthetic_logs(
    count=10_000,       # 10K log records
    duration_hours=2,
    if_not_exists=True,
)
if log_files:
    print(f"Written {len(log_files)} log files")

print("\nData generation complete.")

## 4. Load and Explore Spans

Load span data with time-based filtering and partition pruning.

In [ ]:
# Define time range
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=2)

print(f"Loading spans from {start_time} to {end_time}")

# Load spans (Dask DataFrame - lazy)
spans_ddf = dataset.load_spans(
    start_time=start_time,
    end_time=end_time,
)

print(f"Partitions: {spans_ddf.npartitions}")
print(f"Columns: {list(spans_ddf.columns)}")

In [ ]:
# Compute to see the data (triggers execution)
spans_df = spans_ddf.compute()
print(f"Loaded {len(spans_df):,} spans")
spans_df.head()

In [ ]:
# List available services
services = dataset.list_services(start_time, end_time)
print(f"Services: {services}")

In [ ]:
# Get summary statistics
stats = dataset.get_statistics(start_time, end_time)
stats

## 5. Trace Visualizations

Use the TraceVisualizer for trace-specific visualizations.

In [ ]:
from cybersec.observability.viz import TraceVisualizer

trace_viz = TraceVisualizer(dataset)

In [ ]:
# Latency heatmap by service over time
heatmap = trace_viz.latency_heatmap(
    spans_df,
    y_dim='service_name',
    aggregator='p99',
    width=900,
    height=400,
)
heatmap

In [ ]:
# Latency distribution by service
distribution = trace_viz.latency_distribution(
    spans_df,
    group_by='service_name',
)
distribution

In [ ]:
# Span count time series
timeseries = trace_viz.span_count_timeseries(
    spans_df,
    interval='5min',
    group_by='service_name',
)
timeseries

In [ ]:
# Flame graph (aggregated time per operation)
flame = trace_viz.flame_graph(
    spans_df,
    aggregator='duration_sum',
)
flame

In [ ]:
# Error timeline
errors = trace_viz.error_timeline(spans_df)
errors

### View Single Trace

Get a specific trace ID and view its waterfall diagram.

In [ ]:
# Get a sample trace ID
sample_trace_id = spans_df['trace_id'].iloc[0]
print(f"Viewing trace: {sample_trace_id}")

# Waterfall diagram
waterfall = trace_viz.waterfall(trace_id=sample_trace_id)
waterfall

## 6. Service Topology Visualization

Visualize service dependencies and call relationships.

In [ ]:
from cybersec.observability.viz import TopologyVisualizer

topo_viz = TopologyVisualizer(dataset)

In [ ]:
# Service dependency graph
service_graph = topo_viz.service_graph(
    spans_df,
    layout='spring',
    width=800,
    height=600,
)
service_graph

In [ ]:
# Dependency matrix (heatmap)
dep_matrix = topo_viz.dependency_matrix(
    spans_df,
    metric='call_count',
)
dep_matrix

In [ ]:
# Service health dashboard
health = topo_viz.service_health_dashboard(
    start_time=start_time,
    end_time=end_time,
)
health

## 7. Metrics Visualization

In [ ]:
# Load metrics
metrics_ddf = dataset.load_metrics(
    start_time=start_time,
    end_time=end_time,
)

metrics_df = metrics_ddf.compute()
print(f"Loaded {len(metrics_df):,} metric data points")
metrics_df.head()

In [ ]:
from cybersec.observability.viz import MetricsVisualizer

metrics_viz = MetricsVisualizer(dataset)

In [ ]:
# List available metrics
metric_names = dataset.list_metric_names(start_time, end_time)
print(f"Available metrics: {metric_names}")

In [ ]:
# Time series for a specific metric
if metric_names:
    ts_plot = metrics_viz.timeseries(
        metric_name=metric_names[0],
        start_time=start_time,
        end_time=end_time,
        group_by='service_name',
    )
    display(ts_plot)

## 8. Data Transforms

Use transform functions for custom analysis.

In [ ]:
from cybersec.observability.transforms import (
    add_derived_columns,
    compute_service_metrics,
    extract_service_dependencies,
)

In [ ]:
# Add derived columns (datetime, duration_ms, is_error)
enriched_df = add_derived_columns(spans_df)
enriched_df[['start_time', 'duration_ms', 'is_error']].head()

In [ ]:
# Compute service metrics
service_metrics = compute_service_metrics(spans_df)
service_metrics

In [ ]:
# Extract service dependencies
dependencies = extract_service_dependencies(spans_df)
dependencies

## 9. Interactive Explorer

Launch the full interactive Panel dashboard.

In [ ]:
from cybersec.observability.viz.panels import OTelExplorer

explorer = OTelExplorer(dataset)

# Show in notebook
explorer.notebook()

In [ ]:
# Or launch as standalone app (opens in browser)
# explorer.show()

## 10. Cleanup

In [ ]:
# Close Dask client
client.close()

---

## Summary

This notebook demonstrated:

1. **OTelDataset**: Out-of-core loading with partition pruning and predicate pushdown
2. **OTelWriter**: Generating and writing synthetic OTel data to Parquet
3. **TraceVisualizer**: Waterfall, heatmap, flame graph, and distribution charts
4. **TopologyVisualizer**: Service dependency graphs and health dashboards
5. **MetricsVisualizer**: Time series and metric comparisons
6. **Transforms**: Service metrics, dependencies, and derived columns
7. **OTelExplorer**: Interactive Panel dashboard

### Next Steps

- Connect to production OTEL Collector for real data
- Set up scheduled Parquet compaction for optimal query performance
- Configure alerting based on service metrics
- Deploy Panel dashboard as standalone service